# Calibration ML Walk-Forward

Notebook-first ML workflow for daily high-temperature calibration. This notebook keeps the point-in-time rules strict: each validation contract date is predicted using only earlier contract dates.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CALIBRATION_DIR = PROJECT_ROOT / "data" / "calibration"
CALIBRATION_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT, CALIBRATION_DIR

In [ ]:
import pandas as pd

from src.calibration.dataset import build_calibration_samples
from src.calibration.modeling import (
    WalkForwardConfig,
    recommended_rules,
    summarize_prediction_results,
    walk_forward_baseline_predictions,
    walk_forward_ml_predictions,
)
from src.calibration.report import write_report

## Load Or Build Samples

This reads local actuals, local NBM rows, exact NWS captures if present, and any Mostly Right GFS/HRRR cache rows. The legacy NBM rows are relabeled as `provider=nbm`, not `nws`.

In [ ]:
samples_path = CALIBRATION_DIR / "calibration_samples.csv"
if samples_path.exists():
    samples = pd.read_csv(samples_path)
else:
    samples = build_calibration_samples(PROJECT_ROOT, CALIBRATION_DIR)

samples.shape, sorted(samples["provider"].dropna().unique())

In [ ]:
samples.head()

## Walk-Forward Settings

`ml_refit_days=30` means each ML model is refit every 30 validation dates. Predictions remain point-in-time safe because every block trains only on dates earlier than the block start.

In [ ]:
config = WalkForwardConfig(
    min_train_days=90,
    shrinkage_k=30.0,
    ml_refit_days=30,
)
config

## Baseline Reference

Baselines are kept in the notebook so ML is always compared against the same walk-forward splits.

In [ ]:
baseline_predictions = walk_forward_baseline_predictions(samples, config)
baseline_results = summarize_prediction_results(baseline_predictions, model_family="baseline")
baseline_results.to_csv(CALIBRATION_DIR / "baseline_results.csv", index=False)

baseline_results.loc[baseline_results["evaluation_scope"].eq("overall")].sort_values("mae_after_f")

## ML Models

The ML candidate set is intentionally conservative: Ridge, ElasticNet, RandomForest, ExtraTrees, and HistGradientBoosting. LightGBM/XGBoost are not required.

In [ ]:
ml_predictions = walk_forward_ml_predictions(samples, config)
ml_results = summarize_prediction_results(ml_predictions, model_family="ml")
ml_results.to_csv(CALIBRATION_DIR / "ml_results.csv", index=False)

ml_results.loc[ml_results["evaluation_scope"].eq("overall")].sort_values("mae_after_f")

## Stability Splits

Use these splits to reject ML if it only wins overall but is unstable by station, provider, or month.

In [ ]:
station_month = ml_results.loc[
    ml_results["evaluation_scope"].eq("station_provider_month")
].copy()
station_month.sort_values(["method", "mae_improvement_f"]).head(20)

## Write Rules And Report

The rules prioritize robust shrinkage unless ML clears the improvement and stability thresholds.

In [ ]:
rules = recommended_rules(samples, baseline_predictions, baseline_results, ml_results)
rules.to_csv(CALIBRATION_DIR / "recommended_calibration_rules.csv", index=False)
report_path = write_report(CALIBRATION_DIR)

rules.head(), report_path

In [ ]:
print(report_path.read_text(encoding="utf-8"))